# 01 — Data Integration & Enrichment

**Objective**: Load, clean, and enrich BAC exam data with socio-economic indicators

**Inputs**: 
- `data/bac_2018_2025.csv` - BAC results by governorate
- `data/ecoles.csv` - School infrastructure data
- `/data/socio_economic.csv` Socio-economic indicators (poverty, illiteracy, population)

**Output**: `outputs/enriched_bac_data.csv` - Integrated dataset ready for analysis

---

In [29]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import os
os.makedirs('../../outputs', exist_ok=True)
os.makedirs('../../viz', exist_ok=True)

print('✅ Imports OK')

✅ Imports OK


## 1. Load BAC Data

In [30]:
# Load BAC data
bac = pd.read_csv('../data/bac_2018_2025.csv', encoding='utf-8')
print(f'BAC data shape: {bac.shape}')
print(f'Governorates: {bac.shape[0]}')
print(f'Columns: {bac.shape[1]}')
display(bac.head(3))

BAC data shape: (26, 34)
Governorates: 26
Columns: 34


,Governorate,Code,2018_Rank,2018_Present,2018_Pass,2018_Rate,2019_Rank,2019_Present,2019_Pass,2019_Rate,...,2023_Pass,2023_Rate,2024_Rank,2024_Present,2024_Pass,2024_Rate,2025_Rank,2025_Present,2025_Pass,2025_Rate
0,صفاقس 1,71,1,4771,2666,"55,88%",3,6071,2611,"43,01%",...,2984,56.05%,2,5354,3353,"62,63%",1,5840,3256,55.75%
1,صفاقس 2,72,2,2800,1359,"48,54%",1,3070,1514,"49,32%",...,1895,51.31%,1,3791,2403,63.39%,2,4307,2364,54.89%
2,المنستير,83,4,5654,2519,"44,55%",2,6551,2845,"43,43%",...,2928,48.48%,4,3554,2067,58.16%,10,6928,3161,45.63%


## 2. Clean BAC Rates (Parse Percentages)

In [31]:
# Years in dataset
YEARS = [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

# Clean rate columns: "64,36%" → 64.36
for y in YEARS:
    col = f'{y}_Rate'
    if col in bac.columns:
        bac[col] = (
            bac[col].astype(str)
            .str.replace(',', '.', regex=False)
            .str.replace('%', '', regex=False)
            .str.strip()
            .astype(float)
        )

print('✅ BAC rates cleaned')
#print(bac[[f'{y}_Rate' for y in YEARS[:3]]].head())

✅ BAC rates cleaned


## 3. Map Governorate Names (Arabic → French)

In [32]:
# Mapping Arabic governorate names to French
GOV_MAP = {
    'صفاقس 1':    'Sfax 1',    'صفاقس 2':    'Sfax 2',
    'المنستير':   'Monastir',  'المهدية':    'Mahdia',
    'سوسة':       'Sousse',    'أريانة':     'Ariana',
    'نابل':       'Nabeul',    'تونس 1':     'Tunis 1',
    'مدنين':      'Médenine',  'تونس 2':     'Tunis 2',
    'بن عروس':    'Ben Arous', 'بنزرت':      'Bizerte',
    'قابس':       'Gabès',     'منوبة':      'Mannouba',
    'زغوان':      'Zaghouan',  'سليانة':     'Siliana',
    'باجة':       'Béja',      'الكاف':      'Kef',
    'القيروان':   'Kairouan',  'القصرين':    'Kasserine',
    'سيدي بوزيد': 'Sidi Bouzid','توزر':      'Tozeur',
    'تطاوين':     'Tataouine', 'قبلي':       'Kébili',
    'قفصة':       'Gafsa',     'جندوبة':     'Jendouba',
}

bac['gov_fr'] = bac['Governorate'].map(GOV_MAP)
unmapped = bac[bac['gov_fr'].isna()]['Governorate'].unique()
if len(unmapped) > 0:
    print(f'⚠️ Unmapped governorates: {unmapped}')
else:
    print('✅ All governorates mapped')

print(f'Total governorates: {bac["gov_fr"].nunique()}')

✅ All governorates mapped
Total governorates: 26


## 4. Load School Infrastructure Data

In [33]:
# Load schools data
ecoles = pd.read_csv('../data/ecoles.csv', encoding='utf-8')
print(f'Schools data shape: {ecoles.shape}')
ecoles.head()

Schools data shape: (6109, 14)


,_id,code_etablissement,CRE,delegation,nom_etablissement,Type,cre_ar,delagation_ar,code_auto,nom_etablissement_ar,region_ar,cycle_ar,Latitude initiale,Longitude initiale
0,1,10100101.0,Tunis 1,TUNIS EL MADINA,E.P.BAB JEDID,E.PRIMAIRE,تونس 1,تونس المدينة,100101.0,م.إ. باب الجديد,منطقة بلدية,مدرسة ابتدائية,36.79216,10.173194
1,2,10100104.0,Tunis 1,TUNIS EL MADINA,E.P.RUE EL MARR,E.PRIMAIRE,تونس 1,تونس المدينة,100104.0,م.إ. نهج المر,منطقة بلدية,مدرسة ابتدائية,36.473968,10.1075
2,3,10100107.0,Tunis 1,TUNIS EL MADINA,E.P.ABD AZIZ LASREM,E.PRIMAIRE,تونس 1,تونس المدينة,100107.0,م.إ. عبد العزيز الأصرم,منطقة بلدية,مدرسة ابتدائية,36.796852,10.174021
3,4,10100110.0,Tunis 1,TUNIS EL MADINA,E.P.7 RUE DOCTEUR CASSAR,E.PRIMAIRE,تونس 1,تونس المدينة,100110.0,م.إ. نهج الحكيم كسار,منطقة بلدية,مدرسة ابتدائية,36.80274,10.1699
4,5,10100115.0,Tunis 1,TUNIS EL MADINA,E.P.AP.SADIKI RUE SINAN PACHA,E.PRIMAIRE,تونس 1,تونس المدينة,100115.0,م.إ. الصادقية نهج سنان باشا,منطقة بلدية,مدرسة ابتدائية,36.798828,10.166001


In [34]:
print(ecoles['CRE'].unique())

<StringArray>
[    'Tunis 1',     'Tunis 2',      'Ariana',    'Zaghouan',    'Mannouba',
   'Ben Arous',     'Bizerte',     'bizerte',        'Béja',    'Jendouba',
     'Siliana',         'Kef',   'Kasserine', 'Sidi Bouzid',       'Gafsa',
      'Tozeur',      'Kébili',      'KEBILI',   'Tataouine',    'Médenine',
       'Gabès',       'GABES',      'Sfax 1',      'Sfax 2',      'Mahdia',
    'Kairouan',    'Monastir',      'Sousse',      'Nabeul',    'ZAGHOUAN',
         'KEF',       'GAFSA',      'TOZEUR',      'SFAX 1',      'NABEUL',
           nan]
Length: 36, dtype: str


In [35]:
ecoles['gov_fr'] = ecoles['CRE'].str.strip().str.lower()

ecoles['gov_fr'] = ecoles['gov_fr'].replace({
    'mannouba': 'manouba'
})

ecoles['gov_fr'] = ecoles['gov_fr'].replace({
    'béja': 'beja',
    'kébili': 'kebili',
    'médenine': 'medenine',
    'gabès': 'gabes'
})

In [36]:
# Count schools by type and governorate
ecoles_gouv = (
    ecoles.groupby('gov_fr')
    .agg(
        nb_primaires = ('Type', lambda x: (x == 'E.PRIMAIRE').sum()),
        nb_colleges  = ('Type', lambda x: x.isin(['E.PREP','E.PREP.TECH']).sum()),
        nb_lycees    = ('Type', lambda x: (x == 'LYCEE').sum()),
        nb_total     = ('Type', 'count')
    ).reset_index()
)

print(f'\n✅ Schools aggregated by governorate')
print(f'Governorates with schools: {ecoles_gouv["gov_fr"].nunique()}')
display(ecoles_gouv.head())


✅ Schools aggregated by governorate
Governorates with schools: 26


,gov_fr,nb_primaires,nb_colleges,nb_lycees,nb_total
0,ariana,94,29,22,145
1,beja,153,32,21,206
2,ben arous,156,41,26,223
3,bizerte,207,47,23,277
4,gabes,157,44,25,226


## 5. Load Socio-Economic Data

In [37]:
# Socio-economic indicators (manually compiled from INS Tunisia, 2014 census + recent surveys)
# Load socio-economic data from CSV
socio_df = pd.read_csv('../data/socio_economic.csv')
print(f'✅ Socio-economic data loaded for {len(socio_df)} governorates')
display(socio_df.head())

✅ Socio-economic data loaded for 26 governorates


,gov_fr,poverty_rate,illiteracy_rate,population
0,Sfax 1,8.3,17.1,514182
1,Sfax 2,8.3,17.1,514182
2,Monastir,8.6,11.7,611118
3,Mahdia,17.6,22.2,448273
4,Sousse,12.3,13.6,753670


## 6. Merge All Data

In [38]:
# Mapping ecoles_gouv (lowercase sans accents) -> noms bac (corrects)
ECOLES_TO_BAC = {
    'ariana':      'Ariana',
    'beja':        'Béja',
    'ben arous':   'Ben Arous',
    'bizerte':     'Bizerte',
    'gabes':       'Gabès',
    'gafsa':       'Gafsa',
    'jendouba':    'Jendouba',
    'kairouan':    'Kairouan',
    'kasserine':   'Kasserine',
    'kebili':      'Kébili',
    'kef':         'Kef',
    'mahdia':      'Mahdia',
    'manouba':     'Mannouba',
    'medenine':    'Médenine',
    'monastir':    'Monastir',
    'nabeul':      'Nabeul',
    'sfax 1':      'Sfax 1',
    'sfax 2':      'Sfax 2',
    'sidi bouzid': 'Sidi Bouzid',
    'siliana':     'Siliana',
    'sousse':      'Sousse',
    'tataouine':   'Tataouine',
    'tozeur':      'Tozeur',
    'tunis 1':     'Tunis 1',
    'tunis 2':     'Tunis 2',
    'zaghouan':    'Zaghouan',
}

ecoles_gouv['gov_fr'] = ecoles_gouv['gov_fr'].map(ECOLES_TO_BAC)


In [39]:

# Vérifier
unmapped = ecoles_gouv[ecoles_gouv['gov_fr'].isna()]
if len(unmapped) > 0:
    print(f'⚠️ Non mappés: {unmapped}')
else:
    print('✅ Tous mappés')

✅ Tous mappés


In [40]:
# Merge BAC + schools
enriched = bac.merge(ecoles_gouv, on='gov_fr', how='left')
print(f'After schools merge: {enriched.shape}')

# Merge with socio-economic data
enriched = enriched.merge(socio_df, on='gov_fr', how='left')
print(f'After socio-economic merge: {enriched.shape}')

# On ne garde que nb_lycees — primaires et collèges ne sont pas pertinents pour le BAC
enriched['nb_lycees'] = enriched['nb_lycees'].fillna(0).astype(int)

# Check for missing values
missing = enriched.isnull().sum()
if missing.sum() > 0:
    print(f'\n⚠️ Missing values:\n{missing[missing > 0]}')
else:
    print('\n✅ No missing values')


After schools merge: (26, 39)
After socio-economic merge: (26, 42)

✅ No missing values


In [41]:
# Voir exactement les noms dans chaque source
print("Noms dans bac:")
print(sorted(bac['gov_fr'].unique()))

print("\nNoms dans ecoles_gouv:")
print(sorted(ecoles_gouv['gov_fr'].unique()))

print("\nNoms dans socio_df:")
print(sorted(socio_df['gov_fr'].unique()))

Noms dans bac:
['Ariana', 'Ben Arous', 'Bizerte', 'Béja', 'Gabès', 'Gafsa', 'Jendouba', 'Kairouan', 'Kasserine', 'Kef', 'Kébili', 'Mahdia', 'Mannouba', 'Monastir', 'Médenine', 'Nabeul', 'Sfax 1', 'Sfax 2', 'Sidi Bouzid', 'Siliana', 'Sousse', 'Tataouine', 'Tozeur', 'Tunis 1', 'Tunis 2', 'Zaghouan']

Noms dans ecoles_gouv:
['Ariana', 'Ben Arous', 'Bizerte', 'Béja', 'Gabès', 'Gafsa', 'Jendouba', 'Kairouan', 'Kasserine', 'Kef', 'Kébili', 'Mahdia', 'Mannouba', 'Monastir', 'Médenine', 'Nabeul', 'Sfax 1', 'Sfax 2', 'Sidi Bouzid', 'Siliana', 'Sousse', 'Tataouine', 'Tozeur', 'Tunis 1', 'Tunis 2', 'Zaghouan']

Noms dans socio_df:
['Ariana', 'Ben Arous', 'Bizerte', 'Béja', 'Gabès', 'Gafsa', 'Jendouba', 'Kairouan', 'Kasserine', 'Kef', 'Kébili', 'Mahdia', 'Mannouba', 'Monastir', 'Médenine', 'Nabeul', 'Sfax 1', 'Sfax 2', 'Sidi Bouzid', 'Siliana', 'Sousse', 'Tataouine', 'Tozeur', 'Tunis 1', 'Tunis 2', 'Zaghouan']


## 7. Compute Normalized Features

In [42]:
# Vérifier ce que contient enriched comme colonnes
print(enriched.columns.tolist())

# Vérifier si nb_lycees existe et ce qu'il contient avant le merge
print(enriched[['gov_fr', 'nb_lycees']].to_string())

# Vérifier les noms dans ton dataset établissements
# (le dataframe qui contient nb_lycees avant merge)
print(ecoles['gov_fr'].unique())  # remplace schools_df par ton nom réel

# Vérifier les noms dans enriched
print(enriched['gov_fr'].unique())

['Governorate', 'Code', '2018_Rank', '2018_Present', '2018_Pass', '2018_Rate', '2019_Rank', '2019_Present', '2019_Pass', '2019_Rate', '2020_Rank', '2020_Present', '2020_Pass', '2020_Rate', '2021_Rank', '2021_Present', '2021_Pass', '2021_Rate', '2022_Rank', '2022_Present', '2022_Pass', '2022_Rate', '2023_Rank', '2023_Present', '2023_Pass', '2023_Rate', '2024_Rank', '2024_Present', '2024_Pass', '2024_Rate', '2025_Rank', '2025_Present', '2025_Pass', '2025_Rate', 'gov_fr', 'nb_primaires', 'nb_colleges', 'nb_lycees', 'nb_total', 'poverty_rate', 'illiteracy_rate', 'population']
         gov_fr  nb_lycees
0        Sfax 1         25
1        Sfax 2         19
2      Monastir         28
3        Mahdia         23
4        Sousse         29
5        Ariana         22
6        Nabeul         24
7       Tunis 1         29
8      Médenine         32
9       Tunis 2         18
10    Ben Arous         26
11      Bizerte         23
12        Gabès         25
13     Mannouba         19
14     Zaghouan 

In [43]:
YEARS = [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

# Colonnes present dans le format large
present_cols = [f'{year}_Present' for year in YEARS]

# Moyenne sur les 8 années pour chaque gouvernorat
enriched['mean_present'] = enriched[present_cols].mean(axis=1).round(0).astype(int)

print(enriched[['gov_fr', 'mean_present'] + present_cols].sort_values('mean_present', ascending=False).to_string(index=False))

     gov_fr  mean_present  2018_Present  2019_Present  2020_Present  2021_Present  2022_Present  2023_Present  2024_Present  2025_Present
     Nabeul          7739          6844          8201          8751          7332          6910          7648          7605          8618
  Ben Arous          7232          6609          8006          8365          7219          6969          7227          5580          7883
     Sousse          6906          6462          7650          7850          7018          6187          6801          5652          7629
    Bizerte          5993          5569          6425          5802          5860          5462          5910          6151          6767
    Tunis 1          5982          5732          7416          7508          6023          5276          5455          4496          5947
   Monastir          5882          5654          6551          6646          6112          5570          6039          3554          6928
     Ariana          5803         

In [44]:
# ── Normalisation par population en âge lycée (15–19 ans) ──────────────────
# La population totale inclut enfants, adultes, retraités — peu pertinent.
# On estime la population 15–19 ans : ratio national INS ≈ 7.05% de la population totale.
# Source : INS, Population au 1er Juillet par tranche d'âge, 2023.
# Hypothèse : structure d'âge uniforme entre gouvernorats (meilleure approx disponible).

YOUTH_RATIO = 0.0705  # part des 15-19 ans dans la population nationale (INS 2023)

enriched['pop_15_19_est'] = (enriched['population'] * YOUTH_RATIO).round(0).astype(int)

# Lycées pour 1000 jeunes de 15-19 ans (population cible)
enriched['lycees_per_1k'] = (
    enriched['nb_lycees'] / enriched['pop_15_19_est'] * 1000
).round(3)

print(f'✅ Normalisation par population 15-19 ans (ratio {YOUTH_RATIO*100:.2f}%)')
print(enriched[['gov_fr', 'population', 'pop_15_19_est', 'nb_lycees', 'lycees_per_1k']]
      .sort_values('lycees_per_1k', ascending=False).to_string(index=False))


✅ Normalisation par population 15-19 ans (ratio 7.05%)
     gov_fr  population  pop_15_19_est  nb_lycees  lycees_per_1k
    Siliana      229153          16155         23          1.424
      Gafsa      355341          25052         35          1.397
     Kébili      171478          12089         15          1.241
     Tozeur      116316           8200         10          1.220
  Tataouine      152069          10721         13          1.213
        Kef      247741          17466         19          1.088
       Béja      308710          21764         21          0.965
  Kasserine      465832          32841         30          0.913
Sidi Bouzid      459891          32422         29          0.894
      Gabès      407078          28699         25          0.871
   Médenine      522294          36822         32          0.869
   Zaghouan      191066          13470         11          0.817
    Tunis 1      539206          38014         29          0.763
   Jendouba      405167          28

In [45]:
# Primaires et collèges supprimés : non pertinents pour l'analyse BAC.
print('✅ Normalisation lycees_per_1k calculée dans la cellule précédente.')
print(f'   Plage : [{enriched["lycees_per_1k"].min():.3f}, {enriched["lycees_per_1k"].max():.3f}] lycées / 1000 jeunes 15-19 ans')


✅ Normalisation lycees_per_1k calculée dans la cellule précédente.
   Plage : [0.390, 1.424] lycées / 1000 jeunes 15-19 ans


## 8. Compute Temporal Features (Trend, Volatility)

In [46]:
from scipy import stats

# On utilise les 8 années complètes pour la tendance et la volatilité
HIST_YEARS = [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

# Pente de tendance (régression linéaire du taux sur les années)
def compute_trend(row):
    rates = [row[f'{y}_Rate'] for y in HIST_YEARS]
    slope, *_ = stats.linregress(HIST_YEARS, rates)
    return round(slope, 3)

enriched['rate_trend'] = enriched.apply(compute_trend, axis=1)

# Volatilité (écart-type des taux sur 8 ans)
rate_cols = [f'{y}_Rate' for y in HIST_YEARS]
enriched['rate_volatility'] = enriched[rate_cols].std(axis=1).round(2)

# Taux moyen sur 8 ans
enriched['rate_mean'] = enriched[rate_cols].mean(axis=1).round(2)

print('✅ Temporal features computed (2018–2025)')
print(enriched[['gov_fr', 'rate_mean', 'rate_trend', 'rate_volatility']]
      .sort_values('rate_mean', ascending=False).to_string(index=False))


✅ Temporal features computed (2018–2025)
     gov_fr  rate_mean  rate_trend  rate_volatility
     Sfax 1      54.46       1.796             9.62
     Sfax 2      54.08       1.737             8.54
   Monastir      49.25       1.340             9.07
     Ariana      47.88       1.410             7.27
     Sousse      47.52       1.725             8.94
     Mahdia      46.88       2.243            10.00
    Tunis 1      44.92       2.231             9.29
  Ben Arous      44.70       1.995             8.39
   Médenine      44.44       2.524             9.02
     Nabeul      44.00       1.907             8.71
    Tunis 2      42.11       1.905             9.80
    Bizerte      40.72       1.509             8.07
      Gabès      40.33       2.330            10.14
   Mannouba      37.59       2.121             8.83
       Béja      33.03       1.729             7.29
  Tataouine      32.30       3.634            10.52
   Zaghouan      32.28       1.943             8.03
    Siliana      31.29 

## 9. Data Validation & Quality Checks

In [47]:
# Validation checks
print('📊 Data Validation Report')
print('=' * 50)

YEARS = [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
rate_cols = [f'{y}_Rate' for y in YEARS]

# Check 1: BAC rates in valid range [0, 100]
invalid_rates = enriched[rate_cols].apply(
    lambda x: ((x < 0) | (x > 100)).any(), axis=1).sum()
print(f'✅ Invalid BAC rates: {invalid_rates}')

# Check 2: Socio-economic indicators
invalid_poverty    = ((enriched['poverty_rate'] < 0) | (enriched['poverty_rate'] > 100)).sum()
invalid_illiteracy = ((enriched['illiteracy_rate'] < 0) | (enriched['illiteracy_rate'] > 100)).sum()
print(f'✅ Invalid poverty rates: {invalid_poverty}')
print(f'✅ Invalid illiteracy rates: {invalid_illiteracy}')

# Check 3: Population > 0
invalid_pop = (enriched['population'] <= 0).sum()
print(f'✅ Invalid populations: {invalid_pop}')

# Check 4: Lycées >= 0
invalid_lycees = (enriched['nb_lycees'] < 0).sum()
print(f'✅ Invalid lycée counts: {invalid_lycees}')

# Check 5: lycees_per_1k
print(f'\n📈 Lycées pour 1000 jeunes 15-19 ans:')
print(f'   Min : {enriched["lycees_per_1k"].min():.3f}')
print(f'   Max : {enriched["lycees_per_1k"].max():.3f}')
print(f'   Moyenne : {enriched["lycees_per_1k"].mean():.3f}')

print(f'\n✅ Data validation complete')


📊 Data Validation Report
✅ Invalid BAC rates: 0
✅ Invalid poverty rates: 0
✅ Invalid illiteracy rates: 0
✅ Invalid populations: 0
✅ Invalid lycée counts: 0

📈 Lycées pour 1000 jeunes 15-19 ans:
   Min : 0.390
   Max : 1.424
   Moyenne : 0.816

✅ Data validation complete


## 10. Export Enriched Dataset

In [48]:
YEARS = [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
rate_cols = [f'{y}_Rate' for y in YEARS]

# Colonnes exportées — épurées :
# - nb_primaires, nb_colleges, nb_total : supprimés (non pertinents pour le BAC)
# - primaires_per_1k, colleges_per_1k, schools_per_1k : supprimés
# - pop_15_19_est : conservé pour traçabilité
# - lycees_per_1k : normalisé par population 15-19 ans (INS 2023, ratio 7.05%)
export_cols = [
    'gov_fr', 'Code', 'population', 'pop_15_19_est',
    'nb_lycees', 'lycees_per_1k',
    'poverty_rate', 'illiteracy_rate',
    'rate_mean', 'rate_trend', 'rate_volatility',
] + rate_cols + [f'{y}_Rank' for y in YEARS]

enriched_export = enriched[export_cols].copy()

enriched_export.to_csv('../outputs/enriched_bac_data.csv', index=False, encoding='utf-8')
print(f'✅ Dataset exporté : outputs/enriched_bac_data.csv')
print(f'   Shape : {enriched_export.shape}')
print(f'   Colonnes ({len(export_cols)}) : {export_cols}')
display(enriched_export.head())


✅ Dataset exporté : outputs/enriched_bac_data.csv
   Shape : (26, 27)
   Colonnes (27) : ['gov_fr', 'Code', 'population', 'pop_15_19_est', 'nb_lycees', 'lycees_per_1k', 'poverty_rate', 'illiteracy_rate', 'rate_mean', 'rate_trend', 'rate_volatility', '2018_Rate', '2019_Rate', '2020_Rate', '2021_Rate', '2022_Rate', '2023_Rate', '2024_Rate', '2025_Rate', '2018_Rank', '2019_Rank', '2020_Rank', '2021_Rank', '2022_Rank', '2023_Rank', '2024_Rank', '2025_Rank']


,gov_fr,Code,population,pop_15_19_est,nb_lycees,lycees_per_1k,poverty_rate,illiteracy_rate,rate_mean,rate_trend,...,2024_Rate,2025_Rate,2018_Rank,2019_Rank,2020_Rank,2021_Rank,2022_Rank,2023_Rank,2024_Rank,2025_Rank
0,Sfax 1,71,514182,36250,25,0.690,8.3,17.1,54.46,1.796,...,62.63,55.75,1,3,2,2,1,1,2,1
1,Sfax 2,72,514182,36250,19,0.524,8.3,17.1,54.08,1.737,...,63.39,54.89,2,1,1,1,2,2,1,2
2,Monastir,83,611118,43084,28,0.650,8.6,11.7,49.25,1.340,...,58.16,45.63,4,2,3,3,3,7,4,10
3,Mahdia,82,448273,31603,23,0.728,17.6,22.2,46.88,2.243,...,55.75,47.12,10,6,6,4,6,5,3,5
4,Sousse,84,753670,53134,29,0.546,12.3,13.6,47.52,1.725,...,53.73,46.68,5,5,5,5,4,3,5,7


## Summary

✅ **Data Integration Complete**

- Loaded BAC data (2018-2025) for 26 governorates
- Loaded school infrastructure data (6000+ schools)
- Loaded socio-economic indicators (poverty, illiteracy, population)
- Merged all datasets by governorate
- Computed normalized features (schools per 1000 inhabitants)
- Computed temporal features (trend, volatility)
- Validated data integrity
- Exported enriched dataset to `outputs/enriched_bac_data.csv`

**Next Step**: Run `02_exploratory_analysis.ipynb` to discover patterns and correlations